In [0]:
import pyspark
from pyspark.sql import SparkSession
#from pyspark.sql.functions import col, desc, row_number, rank, dense_rank, sum 
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
df = spark.read.csv(
    "s3://sinac-bronze/2008_2013/sinac2008DatosAbiertos.csv",
    header = True,
    inferSchema = True)

In [0]:
df.show(2)

In [0]:
df.printSchema()

In [0]:
# Amount of births per state with ranking
df_states = df.groupBy('edo_captura').count().orderBy('count', ascending = False)

# rank the states by the count
window_rank = Window.orderBy(F.desc('count'))

#Apply window function to the ranked states
ranked_df_states = df_states.withColumn("Ranking", F.dense_rank().over(window_rank))

ranked_df_states.show(32)


In [0]:
df.select('procedimiento_utilizado').distinct().show()

In [0]:
# Group the entries by how many procedures were made 
df_procedures = df.groupBy('edo_captura', 'procedimiento_utilizado').count().orderBy('edo_captura', "procedimiento_utilizado")

# Add a window function to partition the data 
window_spec = Window.partitionBy('edo_captura')

df_procedures = df_procedures.withColumnRenamed('count', 'procedimiento_conteo' )

# Add the column with the sum for each state
df_procedures = df_procedures.withColumn(
    'edo_conteo',
    F.sum('procedimiento_conteo').over(window_spec)
)

# Add the column with the percentage for each procedure
df_procedures = df_procedures.withColumn(
    'porcentaje_prevalencia',
    F.concat(F.format_number((F.col('procedimiento_conteo') / F.col('edo_conteo')) * 100, 2), F.lit('%'))
)

# Show the data 
df_procedures.show(n=1000, truncate=False)

In [0]:

# First, create your grouped dataframe
df_procedures = df.groupBy('edo_captura', 'procedimiento_utilizado').count()

# Add a window to calculate total per state
window_spec = Window.partitionBy('edo_captura')

# Add the total count per state
df_procedures = df_procedures.withColumn(
    'total_edo_count', 
    F.sum('count').over(window_spec)
).orderBy('edo_captura', 'procedimiento_utilizado')

df_procedures.show()

In [0]:
type(df_procedures.columns)

### Reporte de valores nulos 

- Reportar valores nulos para las columnas
- Reportar como se registran las cesáreas
- Report null-like values (say writing 999 for age instead of null)


Los problemas actuales son que la fecha de nacimiento de la madre parece ser un string y no realmente una fecha entonces tiene valores de 99/99/9999. Esto dicho spark no parece procesar máximos y mínimos confechas, entonces o checamos eso o realizamos una visualización para asegurarnos de que no haya picos raros en los datos. 
Falta borrar los datos máximos de las columnas que los tienen y añadirlo al reporte. Poner logging para todo esto suena a lo mejor pero también una hueva que Dios mío. 

In [0]:
df_base = spark.read.csv(
    "s3://sinac-bronze/2008_2013/sinac2008DatosAbiertos.csv",
    header = True,
    inferSchema = True)

In [0]:
def checkNulls(df: pyspark.sql.connect.dataframe.DataFrame, cols:list[str] = [], delete_records=False, file_path=None) -> pyspark.sql.connect.dataframe.DataFrame:
    """
    Descripcion:
        Regresa un reporte de la cantidad de valores nulos dentro del DataFrame, por defecto de todas las columnas y con funcionalidad para escribir a un reporte de texto.
    
    Parametros:
        df: El DataFrame de pyspark para realizar el análisis
        cols: Las columnas dentro del DataFrame para analizarse 
        delete_records: Si hay un valor nulo en las columnas, si se borra o no el registro
        write_file: Si se escribe el reporte a un archivo de texto.
    """
    if cols == []:
        null_counts = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns])

    return null_counts


In [0]:
df_null = checkNulls(df_base)
df_null.show()

In [0]:
df_base.filter(F.col("localidad_nacimiento").isNull()).count()

### Columnas con valores no strings

In [0]:
# Get only columns that are integers 
int_cols = [(i, col, dtype) for i, (col, dtype) in enumerate(df.dtypes) if dtype == 'int']
print(f'There are {len(int_cols)} integer columns in the dataframe')

for index, col_name, col_type in int_cols:
    print(f'|-- Index {index}: {col_name}')

In [0]:
# Obtengamos unicamente las columnas que no son strings 
numeric_cols = [(i, col, dtype) for i, (col, dtype) in enumerate(df.dtypes) if dtype != 'string']
print(f'There are {len(int_cols)} non string columns in the dataframe')

for index, col_name, col_type in numeric_cols:
    print(f'|-- Index {index}: {col_name}, dtype: {col_type}')

In [0]:
# Creemos un dataframe que contenga unicamente las columnas que no son strings basado en nuestra lista anterior

numeric_cols_names = [sublist[1] for sublist in numeric_cols]

df_numeric = df_base.select(*numeric_cols_names)
df_numeric.show()

In [0]:
df_numeric.summary('max').show()

In [0]:
df_base.describe().show()

## Valores máximos sin sentido 
Los valores máximos no tienen sentido para las columnas en este caso entonces son un valor nulo que se tiene que cambiar o borrar. 

In [0]:
df_base.summary('max').show()

Explorar los valores "nulos"